Убрать тексты на русском (но не на кабардинском/адыгейском) с помощью регулярок (лангдетект тут не поможет), при этом римские цифры оставляем

In [ ]:
!pip install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 13.5 MB/s eta 0:00:00


In [ ]:
import re
from wordfreq import zipf_frequency
import unicodedata
from collections import Counter
import regex as re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ========= НАСТРОЙКИ =========
RU_SENT_THRESHOLD = 0.7
STICKS = set("ӀӏIiⅠⅼ|1")
HYPHENS = "-–—−"

# ========= РЕГУЛЯРКИ =========
WORD_RE = re.compile(
    rf"(?:\p{{Cyrillic}}|\p{{Mn}}|\p{{Nd}}|[{HYPHENS}])+",
    re.IGNORECASE
)

SENT_RE = re.compile(r"[^.!?]+[.!?]?", re.DOTALL)

RU_FUNCTION_WORDS = {
    "и", "в", "во", "на", "по", "о", "об", "от", "до",
    "за", "из", "к", "ко", "с", "со",
    "что", "как", "а", "но", "да", "или", "ли",
    "же", "бы", "то", "не", "ни", "мы", "вы", "он",
    "она", "они", "оно", "это", "нет"
}

In [ ]:
# ========= ФАЙЛЫ =========
INPUT_FILE = "/content/drive/MyDrive/Проект/2_Adyghe_NoOCR.txt"
OUTPUT_CLEAN = "/content/drive/MyDrive/Проект/3_Adyghe_NoOCR.txt"
OUTPUT_REMOVED_WORDS = "/content/drive/MyDrive/Проект/3_Adyghe_NoOCR_removed_ru_words.tsv"

# ========= ФУНКЦИИ =========
def letter_len(word):
    return sum(
        1 for ch in word
        if unicodedata.category(ch).startswith("L")
    )

def is_ru_word(word):
    if any(ch in STICKS for ch in word):
        return False
    if letter_len(word) <= 3:
        return False
    word_nfc = unicodedata.normalize("NFC", word)
    if word_nfc.lower() in RU_FUNCTION_WORDS:
        return False
    return zipf_frequency(word_nfc.lower(), "ru") > 0

# ========= ЧТЕНИЕ ТЕКСТА =========
with open(INPUT_FILE, encoding="utf-8") as f:
    text = f.read()

removed_counter = Counter()
clean_sentences = []

# ========= ОБРАБОТКА =========
for sent in SENT_RE.findall(text):
    words = WORD_RE.findall(sent)

    ru_words = []
    content_words = []

    for w in words:
        if letter_len(w) <= 3:
            continue
        content_words.append(w)
        if is_ru_word(w):
            ru_words.append(w)

    # ---- если предложение почти полностью русское ----
    if content_words and (len(ru_words) / len(content_words) >= RU_SENT_THRESHOLD):
        for w in ru_words:
            removed_counter[w] += 1
        continue  # не добавляем предложение в текст

    # ---- иначе удаляем только русские слова ----
    def remove_ru(match):
        word = match.group(0)
        if is_ru_word(word):
            removed_counter[word] += 1
            return ""
        return word

    cleaned_sent = WORD_RE.sub(remove_ru, sent)
    clean_sentences.append(cleaned_sent)

clean_text = "".join(clean_sentences)

# ========= ПРОВЕРКА NFD =========
assert unicodedata.normalize("NFD", clean_text) == clean_text

# ========= СОХРАНЕНИЕ =========
with open(OUTPUT_CLEAN, "w", encoding="utf-8") as f:
    f.write(clean_text)

with open(OUTPUT_REMOVED_WORDS, "w", encoding="utf-8") as f:
    f.write("word\tcount\n")
    for word, cnt in removed_counter.most_common():
        f.write(f"{word}\t{cnt}\n")

# ========= СТАТИСТИКА =========
total_removed = sum(removed_counter.values())
unique_removed = len(removed_counter)

print("ГОТОВО")
print("Удалено всего русских слов:", total_removed)
print("Уникальных удалённых слов:", unique_removed)
print("Очищенный текст сохранён в:", OUTPUT_CLEAN)
print("Список удалённых слов сохранён в:", OUTPUT_REMOVED_WORDS)

ГОТОВО
Удалено всего русских слов: 534670
Уникальных удалённых слов: 31962
Очищенный текст сохранён в: /content/drive/MyDrive/Проект/3_Adyghe_NoOCR.txt
Список удалённых слов сохранён в: /content/drive/MyDrive/Проект/3_Adyghe_NoOCR_removed_ru_words.tsv


In [ ]:
# ========= ФАЙЛЫ =========
INPUT_FILE = "/content/drive/MyDrive/Проект/2_Adyghe_OCR.txt"
OUTPUT_CLEAN = "/content/drive/MyDrive/Проект/3_Adyghe_OCR.txt"
OUTPUT_REMOVED_WORDS = "/content/drive/MyDrive/Проект/3_Adyghe_OCR_removed_ru_words.tsv"

# ========= ЧТЕНИЕ ТЕКСТА =========
with open(INPUT_FILE, encoding="utf-8") as f:
    text = f.read()

removed_counter = Counter()
clean_sentences = []

# ========= ОБРАБОТКА =========
for sent in SENT_RE.findall(text):
    words = WORD_RE.findall(sent)

    ru_words = []
    content_words = []

    for w in words:
        if letter_len(w) <= 3:
            continue
        content_words.append(w)
        if is_ru_word(w):
            ru_words.append(w)

    # ---- если предложение почти полностью русское ----
    if content_words and (len(ru_words) / len(content_words) >= RU_SENT_THRESHOLD):
        for w in ru_words:
            removed_counter[w] += 1
        continue  # не добавляем предложение в текст

    # ---- иначе удаляем только русские слова ----
    def remove_ru(match):
        word = match.group(0)
        if is_ru_word(word):
            removed_counter[word] += 1
            return ""
        return word

    cleaned_sent = WORD_RE.sub(remove_ru, sent)
    clean_sentences.append(cleaned_sent)

clean_text = "".join(clean_sentences)

# ========= ПРОВЕРКА NFD =========
assert unicodedata.normalize("NFD", clean_text) == clean_text

# ========= СОХРАНЕНИЕ =========
with open(OUTPUT_CLEAN, "w", encoding="utf-8") as f:
    f.write(clean_text)

with open(OUTPUT_REMOVED_WORDS, "w", encoding="utf-8") as f:
    f.write("word\tcount\n")
    for word, cnt in removed_counter.most_common():
        f.write(f"{word}\t{cnt}\n")

# ========= СТАТИСТИКА =========
total_removed = sum(removed_counter.values())
unique_removed = len(removed_counter)

print("ГОТОВО")
print("Удалено всего русских слов:", total_removed)
print("Уникальных удалённых слов:", unique_removed)
print("Очищенный текст сохранён в:", OUTPUT_CLEAN)
print("Список удалённых слов сохранён в:", OUTPUT_REMOVED_WORDS)

ГОТОВО
Удалено всего русских слов: 537077
Уникальных удалённых слов: 32100
Очищенный текст сохранён в: /content/drive/MyDrive/Проект/3_Adyghe_OCR.txt
Список удалённых слов сохранён в: /content/drive/MyDrive/Проект/3_Adyghe_OCR_removed_ru_words.tsv


In [ ]:
# ========= ФАЙЛЫ =========
INPUT_FILE = "/content/drive/MyDrive/Проект/2_Kabardian_NoOCR.txt"
OUTPUT_CLEAN = "/content/drive/MyDrive/Проект/3_Kabardian_NoOCR.txt"
OUTPUT_REMOVED_WORDS = "/content/drive/MyDrive/Проект/3_Kabardian_NoOCR_removed_ru_words.tsv"

# ========= ЧТЕНИЕ ТЕКСТА =========
with open(INPUT_FILE, encoding="utf-8") as f:
    text = f.read()

removed_counter = Counter()
clean_sentences = []

# ========= ОБРАБОТКА =========
for sent in SENT_RE.findall(text):
    words = WORD_RE.findall(sent)

    ru_words = []
    content_words = []

    for w in words:
        if letter_len(w) <= 3:
            continue
        content_words.append(w)
        if is_ru_word(w):
            ru_words.append(w)

    # ---- если предложение почти полностью русское ----
    if content_words and (len(ru_words) / len(content_words) >= RU_SENT_THRESHOLD):
        for w in ru_words:
            removed_counter[w] += 1
        continue  # не добавляем предложение в текст

    # ---- иначе удаляем только русские слова ----
    def remove_ru(match):
        word = match.group(0)
        if is_ru_word(word):
            removed_counter[word] += 1
            return ""
        return word

    cleaned_sent = WORD_RE.sub(remove_ru, sent)
    clean_sentences.append(cleaned_sent)

clean_text = "".join(clean_sentences)

# ========= ПРОВЕРКА NFD =========
assert unicodedata.normalize("NFD", clean_text) == clean_text

# ========= СОХРАНЕНИЕ =========
with open(OUTPUT_CLEAN, "w", encoding="utf-8") as f:
    f.write(clean_text)

with open(OUTPUT_REMOVED_WORDS, "w", encoding="utf-8") as f:
    f.write("word\tcount\n")
    for word, cnt in removed_counter.most_common():
        f.write(f"{word}\t{cnt}\n")

# ========= СТАТИСТИКА =========
total_removed = sum(removed_counter.values())
unique_removed = len(removed_counter)

print("ГОТОВО")
print("Удалено всего русских слов:", total_removed)
print("Уникальных удалённых слов:", unique_removed)
print("Очищенный текст сохранён в:", OUTPUT_CLEAN)
print("Список удалённых слов сохранён в:", OUTPUT_REMOVED_WORDS)

ГОТОВО
Удалено всего русских слов: 1471865
Уникальных удалённых слов: 108221
Очищенный текст сохранён в: /content/drive/MyDrive/Проект/3_Kabardian_NoOCR.txt
Список удалённых слов сохранён в: /content/drive/MyDrive/Проект/3_Kabardian_NoOCR_removed_ru_words.tsv


In [ ]:
# ========= ФАЙЛЫ =========
INPUT_FILE = "/content/drive/MyDrive/Проект/2_Kabardian_OCR.txt"
OUTPUT_CLEAN = "/content/drive/MyDrive/Проект/3_Kabardian_OCR.txt"
OUTPUT_REMOVED_WORDS = "/content/drive/MyDrive/Проект/3_Kabardian_OCR_removed_ru_words.tsv"

# ========= ЧТЕНИЕ ТЕКСТА =========
with open(INPUT_FILE, encoding="utf-8") as f:
    text = f.read()

removed_counter = Counter()
clean_sentences = []

# ========= ОБРАБОТКА =========
for sent in SENT_RE.findall(text):
    words = WORD_RE.findall(sent)

    ru_words = []
    content_words = []

    for w in words:
        if letter_len(w) <= 3:
            continue
        content_words.append(w)
        if is_ru_word(w):
            ru_words.append(w)

    # ---- если предложение почти полностью русское ----
    if content_words and (len(ru_words) / len(content_words) >= RU_SENT_THRESHOLD):
        for w in ru_words:
            removed_counter[w] += 1
        continue  # не добавляем предложение в текст

    # ---- иначе удаляем только русские слова ----
    def remove_ru(match):
        word = match.group(0)
        if is_ru_word(word):
            removed_counter[word] += 1
            return ""
        return word

    cleaned_sent = WORD_RE.sub(remove_ru, sent)
    clean_sentences.append(cleaned_sent)

clean_text = "".join(clean_sentences)

# ========= ПРОВЕРКА NFD =========
assert unicodedata.normalize("NFD", clean_text) == clean_text

# ========= СОХРАНЕНИЕ =========
with open(OUTPUT_CLEAN, "w", encoding="utf-8") as f:
    f.write(clean_text)

with open(OUTPUT_REMOVED_WORDS, "w", encoding="utf-8") as f:
    f.write("word\tcount\n")
    for word, cnt in removed_counter.most_common():
        f.write(f"{word}\t{cnt}\n")

# ========= СТАТИСТИКА =========
total_removed = sum(removed_counter.values())
unique_removed = len(removed_counter)

print("ГОТОВО")
print("Удалено всего русских слов:", total_removed)
print("Уникальных удалённых слов:", unique_removed)
print("Очищенный текст сохранён в:", OUTPUT_CLEAN)
print("Список удалённых слов сохранён в:", OUTPUT_REMOVED_WORDS)

ГОТОВО
Удалено всего русских слов: 2001335
Уникальных удалённых слов: 141735
Очищенный текст сохранён в: /content/drive/MyDrive/Проект/3_Kabardian_OCR.txt
Список удалённых слов сохранён в: /content/drive/MyDrive/Проект/3_Kabardian_OCR_removed_ru_words.tsv


Подсчет слов

In [ ]:
INPUT_FILE = "/content/drive/MyDrive/Проект/2_Kabardian_OCR.txt"

with open(INPUT_FILE, encoding="utf-8") as f:
    text = f.read()

words = text.split()

print("Всего слов:", len(words))

Всего слов: 33622468
